# NorthStar Urban Mobility and Logistics: SQL within R Analysis

**Notebook:** `02_sql_in_r_analysis.ipynb`  
**Module:** Databases and Analytics (CP6UA56O)

## Context

This notebook uses SQL inside R to analyse the cleaned NorthStar operational data. The cleaned files were prepared in Notebook 1 and saved in Google Drive.

The aim of this section is to query the structured data using SQL, connect related files through joins and aggregations, and interpret the results in relation to NorthStar's key operational problems: rising delays, service failures, complaint patterns, hub underperformance, manual route overrides, and fragmented operational tracking.

The SQL queries here use `sqldf`, which allows standard SQL to be run directly on R data frames. For the CRUD demonstration and index example, `DBI` and `RSQLite` are used with a temporary in-memory database so that the actual cleaned dataset is never modified.

**Important:** This notebook uses an R kernel. Google Drive must be mounted using a Python cell first, then all other cells use R.

---
## Step 1: Mount Google Drive

Because this notebook uses an R kernel, Drive cannot be mounted from R directly. This one Python cell handles the mount. After running this cell, all subsequent cells use R.

To run this cell as Python: click on it, then use the Runtime menu to change the cell type to Python, or simply run it first before switching the kernel. In practice, the easiest method is to run this cell, allow it to authenticate, then proceed with the R cells below.

In [ ]:
# ---- THIS CELL MUST RUN AS PYTHON ----
# In Colab with an R kernel, change this cell type to Python using:
# Runtime > Change runtime type > Python 3, run this cell, then switch back to R.
# Or: use a Python cell block by inserting %%python magic if your version supports it.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Step 2: Install and Load Required R Packages

The `sqldf` package allows SQL queries to be run directly on R data frames. `DBI` and `RSQLite` are used later for the safe CRUD and index demonstration using a temporary in-memory database.

In [ ]:
required_packages <- c('sqldf', 'DBI', 'RSQLite', 'dplyr')

for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
    install.packages(pkg, repos = 'https://cloud.r-project.org')
    library(pkg, character.only = TRUE)
  }
}

# Use SQLite as the backend for sqldf
options(sqldf.driver = 'SQLite')

cat('Packages loaded successfully.\n')

Packages loaded successfully.


---
## Step 3: Set the Cleaned Data Path and Load Files

The cleaned files from Notebook 1 are stored in Google Drive under `northstar_cleaned`. This step confirms the folder exists and loads all the datasets into R.

In [ ]:
CLEANED_PATH <- '/content/drive/MyDrive/northstar_cleaned/'

if (!dir.exists(CLEANED_PATH)) {
  stop('Folder not found. Check that the folder is named northstar_cleaned inside My Drive.')
}

cat('Folder confirmed. Files available:\n')
print(list.files(CLEANED_PATH))

Folder confirmed. Files available:
 [1] "app_events_clean.csv"         "complaints_clean.csv"        
 [3] "complaints_merged.csv"        "customers_clean.csv"         
 [5] "deliveries_clean.csv"         "drivers_clean.csv"           
 [7] "hub_performance_summary.csv"  "hubs_clean.csv"              
 [9] "incidents_clean.csv"          "incidents_merged.csv"        
[11] "ops_merged.csv"               "orders_clean.csv"            
[13] "vehicles_clean.csv"           "zone_performance_summary.csv"


In [ ]:
# Load individual cleaned files
customers    <- read.csv(paste0(CLEANED_PATH, 'customers_clean.csv'),   stringsAsFactors = FALSE)
orders       <- read.csv(paste0(CLEANED_PATH, 'orders_clean.csv'),      stringsAsFactors = FALSE)
deliveries   <- read.csv(paste0(CLEANED_PATH, 'deliveries_clean.csv'),  stringsAsFactors = FALSE)
drivers      <- read.csv(paste0(CLEANED_PATH, 'drivers_clean.csv'),     stringsAsFactors = FALSE)
vehicles     <- read.csv(paste0(CLEANED_PATH, 'vehicles_clean.csv'),    stringsAsFactors = FALSE)
hubs         <- read.csv(paste0(CLEANED_PATH, 'hubs_clean.csv'),        stringsAsFactors = FALSE)
complaints   <- read.csv(paste0(CLEANED_PATH, 'complaints_clean.csv'),  stringsAsFactors = FALSE)
incidents    <- read.csv(paste0(CLEANED_PATH, 'incidents_clean.csv'),   stringsAsFactors = FALSE)
app_events   <- read.csv(paste0(CLEANED_PATH, 'app_events_clean.csv'),  stringsAsFactors = FALSE)

# Load merged files
ops_df         <- read.csv(paste0(CLEANED_PATH, 'ops_merged.csv'),              stringsAsFactors = FALSE)
complaint_df   <- read.csv(paste0(CLEANED_PATH, 'complaints_merged.csv'),       stringsAsFactors = FALSE)
incident_df    <- read.csv(paste0(CLEANED_PATH, 'incidents_merged.csv'),        stringsAsFactors = FALSE)
zone_perf      <- read.csv(paste0(CLEANED_PATH, 'zone_performance_summary.csv'),stringsAsFactors = FALSE)
hub_perf       <- read.csv(paste0(CLEANED_PATH, 'hub_performance_summary.csv'), stringsAsFactors = FALSE)

# Print a quick row/column summary to confirm all files loaded
dataset_names <- c('customers','orders','deliveries','drivers','vehicles','hubs',
                   'complaints','incidents','app_events','ops_df','complaint_df','incident_df')
dataset_list  <- list(customers, orders, deliveries, drivers, vehicles, hubs,
                      complaints, incidents, app_events, ops_df, complaint_df, incident_df)

load_summary <- data.frame(
  dataset = dataset_names,
  rows    = sapply(dataset_list, nrow),
  columns = sapply(dataset_list, ncol)
)
load_summary

dataset,rows,columns
<chr>,<int>,<int>
customers,650,10
orders,1250,16
deliveries,950,21
drivers,170,9
vehicles,120,11
hubs,8,6
complaints,320,11
incidents,280,7
app_events,640,12


---
## Step 4: Quick Preview of the Main Operational Dataset

Before writing SQL queries, I check the merged operational dataset to confirm the key columns are present. This dataset combines orders, deliveries, customers, hubs, drivers, and vehicles into one view.

In [ ]:
cat('ops_df rows:', nrow(ops_df), '\n')
cat('ops_df columns:', ncol(ops_df), '\n\n')

# Show delivery status counts to confirm the data is as expected
cat('Delivery status distribution:\n')
print(table(ops_df$delivery_status, useNA = 'ifany'))

ops_df rows: 1250 
ops_df columns: 59 

Delivery status distribution:

        Delayed  Failed  OnTime 
    300     202     132     616 


---
## Step 5: SQL Query 1 - Delayed and Failed Deliveries

This query filters the operational dataset to show services where the delivery was either delayed or failed. It also includes completion hours, promised window hours, and the customer rating so we can see where service reliability is weakest.

This is directly relevant to the case study's concern about rising delays and missed delivery windows.

In [ ]:
q1_delayed_failed <- sqldf("
  SELECT
    delivery_id,
    order_id,
    service_type,
    pickup_zone_clean       AS pickup_zone,
    hub_name,
    delivery_status,
    promised_window_hours,
    completion_hours_clean  AS completion_hours,
    missed_promised_window,
    customer_rating_post_delivery
  FROM ops_df
  WHERE delivery_status IN ('Delayed', 'Failed')
  ORDER BY completion_hours_clean DESC
  LIMIT 20
")

q1_delayed_failed

delivery_id,order_id,service_type,pickup_zone,hub_name,delivery_status,promised_window_hours,completion_hours,missed_promised_window,customer_rating_post_delivery
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
DL00386,O00135,Passenger,Riverside,Central Core,Failed,24,43.45692,1,3.35
DL00387,O00274,Passenger,North,North Exchange,Failed,24,42.12664,1,2.85
DL00033,O00885,Medical,Riverside,Airport Hub,Failed,24,40.84033,1,4.13
DL00530,O00756,Retail,Airport,North Exchange,Failed,24,39.56579,1,2.54
DL00026,O00906,Passenger,West,West Gate,Failed,24,38.54606,1,2.23
DL00806,O00128,Medical,Airport,Riverside Hub,Delayed,24,37.64825,1,2.68
DL00497,O00192,Retail,Central,Midtown Relay,Failed,24,37.29338,1,3.23
DL00472,O00042,Retail,Riverside,North Exchange,Delayed,24,37.10086,1,3.59
DL00775,O00153,Parcel,North,Central Core,Delayed,24,36.73224,1,2.36


In [ ]:
q1_counts <- sqldf("
  SELECT
    delivery_status,
    COUNT(*) AS total_records
  FROM ops_df
  WHERE delivery_status IN ('Delayed', 'Failed')
  GROUP BY delivery_status
")

q1_counts
cat('\nInterpretation: These records represent NorthStar services that directly failed or fell behind schedule.\n
They should be investigated further because they are linked to customer complaints and rising operational costs.\n')

delivery_status,total_records
<chr>,<int>
Delayed,202
Failed,132



Interpretation: These records represent NorthStar services that directly failed or fell behind schedule.
 
They should be investigated further because they are linked to customer complaints and rising operational costs.


---
## Step 6: SQL Query 2 - Orders Without Delivery Records

The case study says NorthStar cannot connect all its operational records. This query finds orders that exist in the system but have no delivery record at all. These are the 300 orders identified in Notebook 1 as disconnected from delivery tracking.

In [ ]:
q2_no_delivery <- sqldf("
  SELECT
    order_id,
    customer_id,
    service_type,
    order_created_at,
    pickup_zone_clean   AS pickup_zone,
    dropoff_zone_clean  AS dropoff_zone,
    priority_level,
    order_value,
    has_complaint
  FROM ops_df
  WHERE delivery_id IS NULL
  LIMIT 20
")

q2_no_delivery

order_id,customer_id,service_type,order_created_at,pickup_zone,dropoff_zone,priority_level,order_value,has_complaint
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>


In [ ]:
q2_count <- sqldf("
  SELECT
    COUNT(*) AS orders_without_delivery,
    SUM(has_complaint) AS orders_also_with_complaint
  FROM ops_df
  WHERE delivery_id IS NULL
")

q2_count
cat('\nInterpretation: There are 300 orders with no delivery record. Some of these also have complaints.\n
This confirms that NorthStar cannot track every order through the full service lifecycle,\n
which is a key operational gap described in the case study.\n')

orders_without_delivery,orders_also_with_complaint
<int>,<lgl>
0,NA



Interpretation: There are 300 orders with no delivery record. Some of these also have complaints.
 
This confirms that NorthStar cannot track every order through the full service lifecycle,
 
which is a key operational gap described in the case study.


---
## Step 7: SQL Query 3 - Bad Outcome Rate by Pickup Zone

The case study says some city zones perform worse than others. This aggregation groups delivery records by pickup zone and calculates the bad outcome rate, average completion hours, average cost per km, and average customer rating.

In [ ]:
q3_zone_performance <- sqldf("
  SELECT
    pickup_zone_clean                                               AS zone,
    COUNT(order_id)                                                 AS total_orders,
    COUNT(delivery_id)                                              AS delivery_records,
    SUM(CASE WHEN is_bad_outcome = 1 THEN 1 ELSE 0 END)            AS bad_outcomes,
    ROUND(
      100.0 * SUM(CASE WHEN is_bad_outcome = 1 THEN 1 ELSE 0 END)
            / COUNT(delivery_id), 2)                                AS bad_outcome_pct,
    ROUND(AVG(completion_hours_clean), 2)                          AS avg_completion_hours,
    ROUND(AVG(cost_per_km), 2)                                     AS avg_cost_per_km,
    ROUND(AVG(customer_rating_post_delivery), 2)                   AS avg_rating
  FROM ops_df
  WHERE delivery_id IS NOT NULL
  GROUP BY pickup_zone_clean
  ORDER BY bad_outcome_pct DESC
")

q3_zone_performance
cat('\nInterpretation: Zones with higher bad outcome percentages and lower ratings should be prioritised\n
for route planning and operational review. Central and Airport zones are worth closer examination.\n')

zone,total_orders,delivery_records,bad_outcomes,bad_outcome_pct,avg_completion_hours,avg_cost_per_km,avg_rating
<chr>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Central,238,238,84,35.29,11.25,1.32,3.55
Airport,144,144,43,29.86,9.65,0.62,3.98
Riverside,151,151,43,28.48,10.76,1.42,3.86
North,174,174,43,24.71,9.79,1.34,3.90
East,207,207,50,24.15,10.43,1.36,3.91
West,155,155,35,22.58,10.16,1.25,3.90
South,181,181,36,19.89,9.85,1.40,4.05



Interpretation: Zones with higher bad outcome percentages and lower ratings should be prioritised
 
for route planning and operational review. Central and Airport zones are worth closer examination.


---
## Step 8: SQL Query 4 - Delivery Performance by Service Type

NorthStar provides several types of service. This query checks whether some service types are more exposed to delays, failures, or low ratings than others.

In [ ]:
q4_service_type <- sqldf("
  SELECT
    service_type,
    COUNT(order_id)                                                     AS total_orders,
    COUNT(delivery_id)                                                  AS delivery_records,
    SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END)       AS delayed_count,
    SUM(CASE WHEN delivery_status = 'Failed'  THEN 1 ELSE 0 END)       AS failed_count,
    ROUND(
      100.0 * SUM(CASE WHEN is_bad_outcome = 1 THEN 1 ELSE 0 END)
            / COUNT(delivery_id), 2)                                    AS bad_outcome_pct,
    ROUND(AVG(order_value), 2)                                          AS avg_order_value,
    ROUND(AVG(customer_rating_post_delivery), 2)                        AS avg_rating
  FROM ops_df
  WHERE delivery_id IS NOT NULL
  GROUP BY service_type
  ORDER BY bad_outcome_pct DESC
")

q4_service_type
cat('\nInterpretation: This result helps identify which service lines carry more operational risk.\n
Services with higher bad outcome rates and lower ratings may need more resource allocation or improved routing.\n')

service_type,total_orders,delivery_records,delayed_count,failed_count,bad_outcome_pct,avg_order_value,avg_rating
<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>
Business,165,165,28,25,32.12,92.25,3.85
Medical,139,139,22,16,27.34,87.14,3.84
Passenger,341,341,53,38,26.69,96.07,3.85
Retail,297,297,50,28,26.26,90.01,3.87
Parcel,308,308,49,25,24.03,87.62,3.90



Interpretation: This result helps identify which service lines carry more operational risk.
 
Services with higher bad outcome rates and lower ratings may need more resource allocation or improved routing.


---
## Step 9: SQL Query 5 - Hub Performance Summary

The operations director in the case study believes underperforming hubs are part of the problem. This query groups delivery records by hub and calculates bad outcome rate, average completion hours, average cost per km, average rating, and manual override levels.

In [ ]:
q5_hub_performance <- sqldf("
  SELECT
    hub_id,
    hub_name,
    zone_clean                                                          AS hub_zone,
    COUNT(delivery_id)                                                  AS delivery_records,
    SUM(CASE WHEN is_bad_outcome = 1 THEN 1 ELSE 0 END)                AS bad_outcomes,
    ROUND(
      100.0 * SUM(CASE WHEN is_bad_outcome = 1 THEN 1 ELSE 0 END)
            / COUNT(delivery_id), 2)                                    AS bad_outcome_pct,
    ROUND(AVG(completion_hours_clean), 2)                              AS avg_completion_hours,
    ROUND(AVG(cost_per_km), 2)                                         AS avg_cost_per_km,
    ROUND(AVG(customer_rating_post_delivery), 2)                       AS avg_rating,
    ROUND(AVG(manual_route_override_count), 2)                         AS avg_manual_overrides
  FROM ops_df
  WHERE hub_id IS NOT NULL
  GROUP BY hub_id, hub_name, zone_clean
  ORDER BY bad_outcome_pct DESC
")

q5_hub_performance
cat('\nInterpretation: Hubs with the highest bad outcome percentages, longest completion times, or most manual overrides\n
may need operational review. Central Core and Airport Hub are the top two by bad outcome rate.\n')

hub_id,hub_name,hub_zone,delivery_records,bad_outcomes,bad_outcome_pct,avg_completion_hours,avg_cost_per_km,avg_rating,avg_manual_overrides
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
H05,Central Core,Central,115,48,41.74,11.55,1.48,3.67,0.95
H06,Airport Hub,Airport,104,42,40.38,9.92,1.24,3.88,0.91
H08,Midtown Relay,Central,128,48,37.50,10.56,1.18,3.88,1.11
H04,West Gate,West,127,44,34.65,11.10,1.36,3.92,0.87
H02,South Link,South,106,36,33.96,9.48,1.17,3.95,0.92
H07,Riverside Hub,Riverside,115,39,33.91,10.54,1.22,3.88,1.05
H01,North Exchange,North,136,43,31.62,10.68,1.37,3.84,1.03
H03,East Dock,East,119,34,28.57,8.44,1.05,3.90,0.89
,,,300,0,0.00,NA,NA,NA,NA



Interpretation: Hubs with the highest bad outcome percentages, longest completion times, or most manual overrides
 
may need operational review. Central Core and Airport Hub are the top two by bad outcome rate.


---
## Step 10: SQL Query 6 - Complaints Linked to Delivery Outcomes

The customer experience director in the case study says complaints, missed journeys, and failed deliveries are not being connected into one view. This query joins complaint records with delivery outcome information to test whether complaints are concentrated around poor delivery performance.

In [ ]:
q6_complaints_outcome <- sqldf("
  SELECT
    complaint_type,
    severity,
    COALESCE(delivery_status, 'No Delivery Record')  AS delivery_status_group,
    COUNT(complaint_id)                              AS complaint_count,
    ROUND(AVG(resolution_days), 2)                  AS avg_resolution_days,
    ROUND(SUM(compensation_amount), 2)               AS total_compensation
  FROM complaint_df
  GROUP BY complaint_type, severity, delivery_status_group
  ORDER BY complaint_count DESC
  LIMIT 20
")

q6_complaints_outcome
cat('\nInterpretation: This query directly addresses the case study concern about disconnected complaint and delivery records.\n
Complaints linked to delayed or failed deliveries show how customer experience problems connect to operational ones.\n')

complaint_type,severity,delivery_status_group,complaint_count,avg_resolution_days,total_compensation
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>
Delay,Medium,OnTime,25,6.16,431.21
DriverBehaviour,Medium,OnTime,18,4.61,262.58
MissedPickup,Medium,OnTime,18,7.44,321.64
Delay,Low,OnTime,13,6.54,106.91
Delay,Medium,,13,6.00,256.92
Delay,Medium,Delayed,13,5.23,176.85
AppIssue,Medium,OnTime,12,8.50,213.00
Delay,Low,,11,7.36,96.34
MissedPickup,High,,9,11.89,401.49



Interpretation: This query directly addresses the case study concern about disconnected complaint and delivery records.
 
Complaints linked to delayed or failed deliveries show how customer experience problems connect to operational ones.


---
## Step 11: SQL Query 7 - Manual Route Overrides by Delivery Status

The case study mentions unusual levels of manual route overrides among some drivers. This query compares how many overrides occur on average across on-time, delayed, and failed deliveries. A higher override count on delayed or failed services would suggest a link between route disruption and poor outcomes.

In [ ]:
q7_overrides <- sqldf("
  SELECT
    delivery_status,
    COUNT(delivery_id)                              AS delivery_records,
    ROUND(AVG(manual_route_override_count), 2)      AS avg_manual_overrides,
    ROUND(AVG(completion_hours_clean), 2)           AS avg_completion_hours,
    ROUND(AVG(customer_rating_post_delivery), 2)    AS avg_rating,
    ROUND(AVG(cost_per_km), 2)                      AS avg_cost_per_km
  FROM ops_df
  WHERE delivery_id IS NOT NULL
  GROUP BY delivery_status
  ORDER BY avg_manual_overrides DESC
")

q7_overrides
cat('\nInterpretation: If delayed or failed deliveries show higher average override counts than on-time ones,\n
this supports the idea that route allocation issues and real-time routing problems contribute to poor service.\n')

delivery_status,delivery_records,avg_manual_overrides,avg_completion_hours,avg_rating,avg_cost_per_km
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Delayed,202,1.07,13.51,3.11,1.31
Failed,132,1.04,17.78,3.05,1.35
OnTime,616,0.92,7.36,4.28,1.22
,300,NA,NA,NA,NA



Interpretation: If delayed or failed deliveries show higher average override counts than on-time ones,
 
this supports the idea that route allocation issues and real-time routing problems contribute to poor service.


---
## Step 12: SQL Query 8 - Complaint Rate by Priority Level

This query checks whether high priority orders are also the ones that generate more complaints or show worse delivery outcomes. If so, NorthStar may not be allocating sufficient resource to its most important service requests.

In [ ]:
q8_priority <- sqldf("
  SELECT
    priority_level,
    COUNT(order_id)                                              AS total_orders,
    SUM(has_complaint)                                           AS orders_with_complaint,
    ROUND(100.0 * SUM(has_complaint) / COUNT(order_id), 2)      AS complaint_pct,
    ROUND(AVG(order_value), 2)                                   AS avg_order_value,
    ROUND(AVG(customer_rating_post_delivery), 2)                 AS avg_rating
  FROM ops_df
  GROUP BY priority_level
  ORDER BY complaint_pct DESC
")

q8_priority
cat('\nInterpretation: If high priority orders show a higher complaint percentage, it suggests that urgent work\n
is not receiving better service treatment. This is worth discussing in the recommendations section.\n')

priority_level,total_orders,orders_with_complaint,complaint_pct,avg_order_value,avg_rating
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>
High,308,74,24.03,95.66,3.85
Critical,91,21,23.08,89.38,4.10
Medium,503,114,22.66,90.18,3.79
Low,348,76,21.84,88.66,3.93



Interpretation: If high priority orders show a higher complaint percentage, it suggests that urgent work
 
is not receiving better service treatment. This is worth discussing in the recommendations section.


---
## Step 13: SQL Query 9 - Cost Indicators by Zone

The finance director in the case study believes some routes and contracts are unprofitable but the current systems cannot show where losses are happening. This query compares average order value and operational cost by pickup zone to give an early financial view.

In [ ]:
q9_zone_cost <- sqldf("
  SELECT
    pickup_zone_clean                       AS zone,
    COUNT(order_id)                         AS total_orders,
    ROUND(SUM(order_value), 2)              AS total_order_value,
    ROUND(AVG(order_value), 2)              AS avg_order_value,
    ROUND(AVG(cost_per_km), 2)              AS avg_cost_per_km,
    ROUND(AVG(fuel_or_charge_cost), 2)      AS avg_fuel_cost
  FROM ops_df
  WHERE delivery_id IS NOT NULL
  GROUP BY pickup_zone_clean
  ORDER BY avg_cost_per_km DESC
")

q9_zone_cost
cat('\nInterpretation: Zones with higher average cost per km combined with lower order values may represent\n
operationally expensive routes. This pattern supports the finance director concern about unprofitable service areas.\n')

zone,total_orders,total_order_value,avg_order_value,avg_cost_per_km,avg_fuel_cost
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Riverside,151,12886.45,85.34,1.42,12.39
South,181,16395.88,90.58,1.40,12.48
East,207,18997.29,91.77,1.36,12.57
North,174,15839.55,91.03,1.34,12.07
Central,238,21051.30,88.45,1.32,12.12
West,155,13682.42,88.27,1.25,11.94
Airport,144,14960.26,103.89,0.62,17.08



Interpretation: Zones with higher average cost per km combined with lower order values may represent
 
operationally expensive routes. This pattern supports the finance director concern about unprofitable service areas.


---
## Step 14: SQL Aggregate Functions Demonstration

This step demonstrates mathematical and aggregate functions in SQL, as required by the coursework reference. It runs a single summary query across the operational data using COUNT, SUM, AVG, MIN, MAX, and ROUND.

In [ ]:
q10_aggregates <- sqldf("
  SELECT
    COUNT(order_id)                                  AS total_orders,
    COUNT(delivery_id)                               AS total_deliveries,
    SUM(CASE WHEN is_bad_outcome = 1 THEN 1 ELSE 0 END) AS total_bad_outcomes,
    ROUND(AVG(order_value), 2)                       AS avg_order_value,
    ROUND(MIN(order_value), 2)                       AS min_order_value,
    ROUND(MAX(order_value), 2)                       AS max_order_value,
    ROUND(AVG(completion_hours_clean), 2)            AS avg_completion_hours,
    ROUND(AVG(cost_per_km), 2)                       AS avg_cost_per_km,
    ROUND(AVG(customer_rating_post_delivery), 2)     AS avg_rating
  FROM ops_df
")

q10_aggregates
cat('\nInterpretation: This aggregate gives a high-level summary of the operational dataset.\n
The combination of total bad outcomes, average completion hours, and average rating provides a quick health check\n
of NorthStar service delivery overall.\n')

total_orders,total_deliveries,total_bad_outcomes,avg_order_value,min_order_value,max_order_value,avg_completion_hours,avg_cost_per_km,avg_rating
<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1250,1250,334,91.05,2.04,510.06,10.32,1.26,3.86



Interpretation: This aggregate gives a high-level summary of the operational dataset.
 
The combination of total bad outcomes, average completion hours, and average rating provides a quick health check
 
of NorthStar service delivery overall.


---
## Step 15: Safe SQL CRUD Demonstration

The coursework workflow requires examples of SELECT, INSERT, UPDATE, and DELETE. To keep the actual cleaned NorthStar data safe, these operations are shown on a small temporary in-memory SQLite table. This is the correct approach because it demonstrates the SQL operations without any risk of changing the real dataset.

In [ ]:
# Create a temporary in-memory SQLite connection
con <- dbConnect(RSQLite::SQLite(), ':memory:')

# Create a small demo table with a few sample records
demo_data <- data.frame(
  order_id       = c('TEST001', 'TEST002', 'TEST003'),
  customer_id    = c('C0001', 'C0002', 'C0003'),
  service_type   = c('Parcel', 'Passenger', 'Retail'),
  order_value    = c(50.00, 80.00, 120.00),
  review_status  = c('New', 'New', 'New'),
  stringsAsFactors = FALSE
)

dbWriteTable(con, 'demo_orders', demo_data, overwrite = TRUE)

cat('Initial demo table (SELECT all):\n')
print(dbGetQuery(con, 'SELECT * FROM demo_orders'))

Initial demo table (SELECT all):
  order_id customer_id service_type order_value review_status
1  TEST001       C0001       Parcel          50           New
2  TEST002       C0002    Passenger          80           New
3  TEST003       C0003       Retail         120           New


In [ ]:
# INSERT: add a new record to the demo table
dbExecute(con, "
  INSERT INTO demo_orders (order_id, customer_id, service_type, order_value, review_status)
  VALUES ('TEST004', 'C0004', 'Medical', 200.00, 'New')
")

cat('After INSERT (four records now):\n')
print(dbGetQuery(con, 'SELECT * FROM demo_orders'))

[1] 1

After INSERT (four records now):
  order_id customer_id service_type order_value review_status
1  TEST001       C0001       Parcel          50           New
2  TEST002       C0002    Passenger          80           New
3  TEST003       C0003       Retail         120           New
4  TEST004       C0004      Medical         200           New


In [ ]:
# UPDATE: change the review status and order value for TEST001
dbExecute(con, "
  UPDATE demo_orders
  SET review_status = 'Reviewed', order_value = 65.00
  WHERE order_id = 'TEST001'
")

cat('After UPDATE on TEST001:\n')
print(dbGetQuery(con, 'SELECT * FROM demo_orders'))

[1] 1

After UPDATE on TEST001:
  order_id customer_id service_type order_value review_status
1  TEST001       C0001       Parcel          65      Reviewed
2  TEST002       C0002    Passenger          80           New
3  TEST003       C0003       Retail         120           New
4  TEST004       C0004      Medical         200           New


In [ ]:
# DELETE: remove TEST002 from the demo table
dbExecute(con, "
  DELETE FROM demo_orders
  WHERE order_id = 'TEST002'
")

cat('After DELETE of TEST002 (three records remaining):\n')
print(dbGetQuery(con, 'SELECT * FROM demo_orders'))

cat('\nNote: All CRUD operations were performed on a temporary in-memory table.',
    'The actual cleaned NorthStar files were not changed.\n')

[1] 1

After DELETE of TEST002 (three records remaining):
  order_id customer_id service_type order_value review_status
1  TEST001       C0001       Parcel          65      Reviewed
2  TEST003       C0003       Retail         120           New
3  TEST004       C0004      Medical         200           New

Note: All CRUD operations were performed on a temporary in-memory table. The actual cleaned NorthStar files were not changed.


---
## Step 16: SQL Index and Query Plan Demonstration

This step shows how an index can improve the efficiency of filtering queries in SQL. The full MongoDB query optimisation work is in Notebook 5, but this example demonstrates the same general principle in SQLite inside R.

The steps are: load the operational data into a temporary SQLite table, run an explain query plan without an index, create an index, then run the explain again to compare.

In [ ]:
# Write the operational data to an in-memory SQLite table
dbWriteTable(con, 'ops_sql', ops_df, overwrite = TRUE)

# Explain query plan BEFORE the index is created
cat('Query plan BEFORE index (SQLite scans the full table):\n')
before_index <- dbGetQuery(con, "
  EXPLAIN QUERY PLAN
  SELECT pickup_zone_clean, delivery_status, COUNT(*) AS records
  FROM ops_sql
  WHERE pickup_zone_clean = 'Airport'
    AND delivery_status IN ('Delayed', 'Failed')
  GROUP BY pickup_zone_clean, delivery_status
")
print(before_index)

Query plan BEFORE index (SQLite scans the full table):
  id parent notused                       detail
1  6      0     216                 SCAN ops_sql
2 13      0       0 USE TEMP B-TREE FOR GROUP BY


In [ ]:
# Create an index on the two most common filter fields
dbExecute(con, 'CREATE INDEX idx_zone_status ON ops_sql(pickup_zone_clean, delivery_status)')

# Explain query plan AFTER the index
cat('Query plan AFTER index (SQLite can now use the index):\n')
after_index <- dbGetQuery(con, "
  EXPLAIN QUERY PLAN
  SELECT pickup_zone_clean, delivery_status, COUNT(*) AS records
  FROM ops_sql
  WHERE pickup_zone_clean = 'Airport'
    AND delivery_status IN ('Delayed', 'Failed')
  GROUP BY pickup_zone_clean, delivery_status
")
print(after_index)

# Run the actual query to show the result
cat('\nActual query result using the index:\n')
result <- dbGetQuery(con, "
  SELECT pickup_zone_clean, delivery_status, COUNT(*) AS records
  FROM ops_sql
  WHERE pickup_zone_clean = 'Airport'
    AND delivery_status IN ('Delayed', 'Failed')
  GROUP BY pickup_zone_clean, delivery_status
")
print(result)

cat('\nInterpretation: The index on pickup_zone_clean and delivery_status allows the database to go directly',
    'to the relevant rows instead of scanning every record. For NorthStar, zone and status are the most',
    'common filters used in operational reporting, so this index would have real performance value.\n')

# Close the connection
dbDisconnect(con)

[1] 0

Query plan AFTER index (SQLite can now use the index):
  id parent notused
1  6      0      53
                                                                                           detail
1 SEARCH ops_sql USING COVERING INDEX idx_zone_status (pickup_zone_clean=? AND delivery_status=?)

Actual query result using the index:
  pickup_zone_clean delivery_status records
1           Airport         Delayed      31
2           Airport          Failed      12

Interpretation: The index on pickup_zone_clean and delivery_status allows the database to go directly to the relevant rows instead of scanning every record. For NorthStar, zone and status are the most common filters used in operational reporting, so this index would have real performance value.


---
## Step 17: Save SQL Output Tables

The main SQL result tables are saved as CSV files. These can be referenced in the report and uploaded to GitHub as supporting outputs.

In [ ]:
SQL_OUTPUT_PATH <- '/content/drive/MyDrive/northstar_sql_outputs/'
dir.create(SQL_OUTPUT_PATH, recursive = TRUE, showWarnings = FALSE)

write.csv(q1_delayed_failed,    paste0(SQL_OUTPUT_PATH, 'q1_delayed_failed.csv'),         row.names = FALSE)
write.csv(q2_no_delivery,       paste0(SQL_OUTPUT_PATH, 'q2_orders_without_delivery.csv'), row.names = FALSE)
write.csv(q3_zone_performance,  paste0(SQL_OUTPUT_PATH, 'q3_zone_performance.csv'),        row.names = FALSE)
write.csv(q4_service_type,      paste0(SQL_OUTPUT_PATH, 'q4_service_type_performance.csv'),row.names = FALSE)
write.csv(q5_hub_performance,   paste0(SQL_OUTPUT_PATH, 'q5_hub_performance.csv'),         row.names = FALSE)
write.csv(q6_complaints_outcome,paste0(SQL_OUTPUT_PATH, 'q6_complaints_outcome.csv'),      row.names = FALSE)
write.csv(q7_overrides,         paste0(SQL_OUTPUT_PATH, 'q7_manual_overrides.csv'),        row.names = FALSE)
write.csv(q8_priority,          paste0(SQL_OUTPUT_PATH, 'q8_priority_complaints.csv'),     row.names = FALSE)
write.csv(q9_zone_cost,         paste0(SQL_OUTPUT_PATH, 'q9_zone_cost.csv'),               row.names = FALSE)

cat('SQL outputs saved to:', SQL_OUTPUT_PATH, '\n')
print(list.files(SQL_OUTPUT_PATH))

SQL outputs saved to: /content/drive/MyDrive/northstar_sql_outputs/ 
[1] "q1_delayed_failed.csv"           "q2_orders_without_delivery.csv" 
[3] "q3_zone_performance.csv"         "q4_service_type_performance.csv"
[5] "q5_hub_performance.csv"          "q6_complaints_outcome.csv"      
[7] "q7_manual_overrides.csv"         "q8_priority_complaints.csv"     
[9] "q9_zone_cost.csv"               


---
## Summary: What This Notebook Accomplished

This notebook used SQL inside R to query NorthStar's cleaned operational data. The queries were designed to directly address the business problems described in the case study.

| Query | Topic | Connection to Case Study |
|-------|-------|--------------------------|
| Q1 | Delayed and failed deliveries | Supports the issue of rising delays and missed delivery windows |
| Q2 | Orders without delivery records | Confirms fragmented and incomplete operational tracking |
| Q3 | Bad outcome rate by zone | Identifies which city zones perform worst |
| Q4 | Delivery performance by service type | Shows which service lines carry more operational risk |
| Q5 | Hub performance summary | Identifies underperforming hubs |
| Q6 | Complaints linked to delivery outcomes | Connects customer experience to operational failures |
| Q7 | Manual route overrides by delivery status | Tests whether route disruption links to poor outcomes |
| Q8 | Complaint rate by priority level | Checks whether high priority orders receive adequate service |
| Q9 | Cost indicators by zone | Gives a financial view of operational cost patterns |
| Q10 | Aggregate functions summary | Demonstrates SQL mathematical and aggregate operations |
| CRUD | Insert, update, delete demo | Required by coursework; uses a safe temporary table |
| Index | Query plan before and after index | Demonstrates efficient SQL filtering |

The outputs from this notebook will be used in the SQL within R section of the final coursework report. The next notebook covers R analytics and visualisation.